In [ ]:
!pip install transformers datasets torchvision sentencepiece
!pip install pdfplumber pdf2image pytesseract docx2txt

In [ ]:
from PIL import Image
import torch
import pandas as pd
import re
import docx2txt
import pytesseract
import pdfplumber
import numpy as np
import json
import spacy
import os


from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer
from pdf2image import convert_from_path
from google.colab import files

In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
filepath = os.path.abspath(filename)

Saving Access Control Policy.docx.pdf to Access Control Policy.docx (1).pdf


In [ ]:
def extract_text_from_pdf(pdf_file):
    """Extract text from PDF file using pdfplumber."""
    extracted_text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            # Extract text while preserving some layout
            extracted_text += page.extract_text() + "\n\n"
    return extracted_text

def extract_text_from_docx(docx_file):
    """Extract text from DOCX file."""
    return docx2txt.process(docx_file)

def extract_text_from_image(image_file, ocr_resolution=300):
    """Extract text from image using OCR."""
    image = Image.open(image_file)
    # Increase resolution for better OCR results
    text = pytesseract.image_to_string(image, config=f'--dpi {ocr_resolution}')
    return text

def extract_text(file_path, file_type):
    """Extract text based on file type."""
    if file_type == "pdf":
        return extract_text_from_pdf(file_path)
    elif file_type in ["docx", "doc"]:
        return extract_text_from_docx(file_path)
    elif file_type in ["jpg", "jpeg", "png"]:
        return extract_text_from_image(file_path)
    elif file_type == "txt":
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        return "Unsupported file format"

def clean_text(text):
    """Clean extracted text for better processing."""
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters that might interfere with model
    text = re.sub(r'[^\w\s.,;:?!()\[\]{}\-\'""`]', '', text)
    return text.strip()

file_extension = filename.split('.')[-1].lower()
extracted_text = extract_text(filepath, file_extension)
cleaned_text = clean_text(extracted_text)
print("Preview:\n", extracted_text[:1000])


Preview:
 Access Control Policy
Access Control Policy
Approved by Signature Effective Date
Security Team Official Security Team 03/11/2025
Approved
1. Purpose
This policy establishes protocols and controls to ensure that access to information,
systems, and resources at DanfeCorp is granted strictly on the basis of business needs
and the principle of least privilege. It is designed to protect sensitive data and company
assets while maintaining a secure operational environment. This document outlines the
procedures for granting, reviewing, and revoking access rights.
2. Scope
This policy applies to all employees, contractors, vendors, and third-party service
providers who require access to DanfeCorp's information systems, networks,
applications, or any resources that process or store sensitive data.
3. Access Control Principles

4. User Access Management
4.1 Account Creation and Provisioning
● New Users: When a new employee or contractor joins DanfeCorp, their role and
responsibilities a

In [ ]:
if "nlp" not in globals():
    try:
        nlp = spacy.load("en_core_web_sm")
    except:
        import subprocess
        subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
        nlp = spacy.load("en_core_web_sm")


In [ ]:
# sentence transformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # Loads a sentence embedding model

In [ ]:
def semantic_segment(text, max_length=1000, similarity_threshold=0.7):
    """
    Segments text into semantically coherent chunks using sentence transformers.
    """
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    if not sentences:
        return []

    embeddings = embedder.encode(sentences, convert_to_tensor=True)
    segments = []
    current_segment = sentences[0]

    for i in range(1, len(sentences)):
        sim = util.cos_sim(embeddings[i - 1], embeddings[i]).item()
        sentence = sentences[i]

        if sim < similarity_threshold or len(current_segment) + len(sentence) > max_length:
            segments.append(current_segment.strip())
            current_segment = sentence
        else:
            current_segment += " " + sentence

    segments.append(current_segment.strip())
    return segments


In [ ]:
segments = semantic_segment(cleaned_text)

df = pd.DataFrame(segments, columns=['segments'])

In [ ]:
df

,segments
0,Access Control Policy Access Control Policy Ap...
1,Purpose This policy establishes protocols and ...
2,It is designed to protect sensitive data and c...
3,This document outlines the procedures for gran...
4,2. Scope
5,"This policy applies to all employees, contract..."
6,3.
7,Access Control Principles 4.
8,User Access Management 4.1 Account Creation an...
9,Role Assignment: Each user is assigned a speci...


Question and Answer Generation

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

print(f"Loading model {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print(f"Model {model_name} loaded successfully!")

Loading model Qwen/Qwen3-0.6B...
Model Qwen/Qwen3-0.6B loaded successfully!


In [ ]:
example_question = [
    "Is access to applications, operating systems, databases, and network devices provisioned according to the principle of least privilege?",
    "Has management approved an access control policy, communicated it to constituents, appointed an owner to maintain it, and reviewed it?",
    "Are remote users prevented from copying data to remote non-corporate devices when using remote terminal services?",
    "Do contractual agreements specify whether third-parties are permitted to resell, assign, or permit access to customer data, or the outsourcer's data, metadata, and systems, to other entities?",
    "Are inactive constituent user IDs disabled and deleted after defined periods of inactivity?"
]

In [ ]:
def build_prompt(text_segment, max_pairs=2):
    formatted_examples = '\n'.join(f"- {q}" for q in example_question)

    return f"""
    You are an expert at analyzing security and compliance documents. Given the following text, generate up to {max_pairs} pairs of relevant question-answer focusing on security and compliance controls, practices, and requirements.

    TEXT:
    {text_segment}

    EXAMPLES:
    {formatted_examples}

    Format your answer like this:
    [
        {{
            "question": "Your question here?",
            "answer": "Answer from the text."
        }},
        ...
    ]
      """


In [ ]:
def generate_from_qwen(prompt, model):
    messages = [
        {"role": "user", "content": prompt}
    ]
    print("Applying chat template...")
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print("Generating text...")
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    print("Output:\n", output_text)

    # Optional: extract only the assistant's answer (after user prompt)
    if "Assistant:" in output_text:
        output_text = output_text.split("Assistant:")[-1].strip()

    return output_text


In [ ]:
def generate_qa_pairs(text_segment):
    qa_pairs = []
    prompt = build_prompt(text_segment)
    response = generate_from_qwen(prompt, model)

    # Use regex to extract questions and answers
    questions = re.findall(r'"question":\s*"([^"]+)"', response)
    answers = re.findall(r'"answer":\s*"([^"]+)"', response)

    # Pair questions with answers
    for i in range(min(len(questions), len(answers))):
        qa_pairs.append({
            "question": questions[i],
            "answer": answers[i]
        })

    return qa_pairs


In [ ]:
df['qa_pairs'] = df['segments'].apply(generate_qa_pairs)

Applying chat template...
Generating text...
Output:
 user

    You are an expert at analyzing security and compliance documents. Given the following text, generate up to 2 pairs of relevant question-answer focusing on security and compliance controls, practices, and requirements.

    TEXT:
    Access Control Policy Access Control Policy Approved by Signature Effective Date Security Team Official Security Team 03112025 Approved 1.

    EXAMPLES:
    - Is access to applications, operating systems, databases, and network devices provisioned according to the principle of least privilege?
- Has management approved an access control policy, communicated it to constituents, appointed an owner to maintain it, and reviewed it?
- Are remote users prevented from copying data to remote non-corporate devices when using remote terminal services?
- Do contractual agreements specify whether third-parties are permitted to resell, assign, or permit access to customer data, or the outsourcer's data, me

In [ ]:
df

In [ ]:
# Extract question and answer from the 'qa_pairs' (which is a list of dictionaries)
df['question'] = df['qa_pairs'].apply(lambda row: row[0]['question'] if isinstance(row, list) and row else '')
df['answer'] = df['qa_pairs'].apply(lambda row: row[0]['answer'] if isinstance(row, list) and row else '')

In [ ]:
df

In [ ]:
df['question']

In [ ]:
# Load a pre-trained transformer model for embeddings (BERT-like model)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
def embedding_text(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    # Use [CLS] token representation as sentence embedding
    cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, hidden_size]
    return cls_embedding.squeeze().numpy()


In [ ]:
# Apply to all paragraphs
df['question_embedding'] = df['question'].apply(embedding_text)
df['answer_embedding'] = df['answer'].apply(embedding_text)

In [ ]:
df

In [ ]:
reference_questions = [
    "Is there an acceptable use policy for information and associated assets?",
    "Does the policy or procedure for information handling include security requirements?",
    "Is there an acceptable use policy for information and associated assets that has been approved by management, communicated to appropriate constituents, and assigned an owner to maintain and periodically review the policy?",
    "Do contractual agreements specify whether third-parties are permitted to resell, assign, or permit access to customer data, or the outsourcer's data, metadata, and systems, to other entities?",
    "Are inactive constituent user IDs disabled within 90 days?",
    "Are inactive constituent user IDs deleted within 120 days?",
    "Does your organization have a process to protect system access from unauthorized boot procedures?",
    "Do IT support personnel have access to application source libraries?",
    "Is authentication required to access mainframe transactions or databases, and do personnel with privileged access to mainframe systems need it?",
    "Are all remote access and file sharing services configured to require authentication and encryption on all servers?",
    "Is the DLP system configured to block unauthorized traffic?",
    "Does the organization have processes and mechanisms to minimize disclosure of internal IP addresses and routing information only to a limited number of authorized parties?",
    "Are there security and hardening standards and secure configurations for network devices, including Firewalls, Switches, Routers and Wireless Access Points?",
    "Is logical network separation of segregated networks enforced using firewalls?",
    "Does the organization require the results of independent reviews to be published?",
    "Does the organization have policies and procedures that mandate reporting to the board or relevant committee regarding the outcomes of utilizing a single provider for various activities?",
    "Has the organization established rules for the safe and appropriate use of online resources, including any restriction to undesirable or inappropriate websites and web-based applications?",
    "Does the Board of Directors for the organization include internal executive and external non-executive directors?",
    "Is the Chairperson of the Board outside the organization?",
    "Is compensation consistent with the organization's code of ethics/conduct or value statements?",
    "Is the organization aware of any breaches in cybersecurity within the last three years?",
    "Is the organization aware of any substantiated complaints regarding breaches of customer privacy or loss of customer data in the last three years?",
    "Does the organization's internal complaints process include a confirmation and discussion with the person making the report, while maintaining confidentiality of their identity and ensuring effective protection against repercussions or punishment as a result of a complaint?"
]

In [ ]:
reference_df = pd.DataFrame(reference_questions, columns=['question'])

In [ ]:
reference_df['question_embedding'] = reference_df['question'].apply(embedding_text)

In [ ]:
reference_df

In [ ]:
similarity_threshold=0.8

In [ ]:
gen_embeddings = np.stack(df['question_embedding'].values)
ref_embeddings = np.stack(reference_df['question_embedding'].values)

# Compute cosine similarity matrix: shape = [num_generated x num_reference]
similarity_matrix = cosine_similarity(gen_embeddings, ref_embeddings)

# For each generated question, get the highest similarity score
max_similarities = similarity_matrix.max()

# Assign these scores back to the 'confidence' column
df['confidence_score'] = max_similarities

In [ ]:
df

In [ ]:
# Compute cosine similarity matrix: shape = [num_generated x num_reference]
similarity_matrix_questions = cosine_similarity(gen_embeddings, gen_embeddings)

# For each generated question, get the highest similarity score
max_similarities = similarity_matrix_questions.min(axis=1)

# Assign these scores back to the 'confidence' column
df['similarity_score'] = max_similarities

In [ ]:
df

In [ ]:
def mark_duplicates(row, df, similarity_threshold, confidence_threshold):
    # Flag to indicate if the row is a duplicate
    is_dup = False

    # Iterate over all rows to compare similarity and confidence scores
    for _, compare_row in df.iterrows():
        # Skip comparing the row with itself
        if row.name == compare_row.name:
            continue

        # Check if the similarity score and confidence score are above the threshold
        if (row['similarity_score'] >= similarity_threshold and
            compare_row['similarity_score'] >= similarity_threshold and
            row['confidence_score'] >= confidence_threshold and
            compare_row['confidence_score'] >= confidence_threshold):
            is_dup = True
            break  # Once a duplicate is found, no need to check further

    return "Duplicate" if is_dup else "Not Duplicate"
# Apply the function to each row to check if it's a duplicate
df['duplicate_flag'] = df.apply(lambda row: mark_duplicates(row, df, similarity_threshold=0.7, confidence_threshold=0.8), axis=1)

# Display the updated DataFrame
print(df)

In [ ]:
df

In [ ]:
def prepare_json(df):
    output = []
    for _, row in df.iterrows():
        output.append({
            "question": row['question'],
            "answer": row['answer'],
            "confidence_score": row['confidence_score']
        })
    return json.dumps(output, indent=4)

# Filter data for non-duplicate rows (if needed)
filtered_df = df[df['duplicate_flag'] == "Not Duplicate"]

# Prepare the filtered data in JSON format
json_output = prepare_json(filtered_df)

In [ ]:
filtered_df

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py
import streamlit as st
def app():
    st.title("Document Question & Answer Extraction")

    # Upload file
    uploaded_file = st.sidebar.file_uploader("Choose a file (PDF, DOCX, JPG, PNG)", type=["pdf", "docx", "jpg", "jpeg", "png"])

    if uploaded_file is not None:
        file_extension = uploaded_file.name.split('.')[-1].lower()
        file_path = os.path.join("/tmp", uploaded_file.name)

        # Save the file and process it
        with open(file_path, "wb") as f:
            f.write(uploaded_file.getbuffer())

        # Process text extraction, segmentation, Q&A generation
        extracted_text = extract_text(file_path, file_extension)
        cleaned_text = clean_text(extracted_text)

        # Segment the text semantically
        segments = semantic_segment(cleaned_text)
        st.subheader("Extracted Segments")
        for i, segment in enumerate(segments[:5], start=1):
            st.text(f"Segment {i}: {segment}")

        # Generate Q&A pairs
        df = pd.DataFrame(segments, columns=['segments'])
        df['qa_pairs'] = df['segments'].apply(generate_qa_pairs)

        # Extract and display question-answer pairs
        df['question'] = df['qa_pairs'].apply(lambda row: row[0]['question'] if isinstance(row, list) and row else '')
        df['answer'] = df['qa_pairs'].apply(lambda row: row[0]['answer'] if isinstance(row, list) and row else '')

        st.subheader("Generated Question-Answer Pairs")
        st.dataframe(df[['question', 'answer']])

        # Generate download button for JSON output
        filtered_df = df[df['duplicate_flag'] == "Not Duplicate"]
        json_output = prepare_json(filtered_df)
        st.download_button(
            label="Download Q&A as JSON",
            data=json_output,
            file_name="qa_pairs.json",
            mime="application/json"
        )
